In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# ML Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [ ]:
# 1. Load and Clean Data
# Adjust the path to match your structure: ../data/raw/cumulative.csv
df = pd.read_csv("../data/raw/cumulative.csv")

# Filter target: Focus on confirmed exoplanets vs false positives
df = df[df['koi_disposition'].isin(['CONFIRMED', 'FALSE POSITIVE'])]

# Feature Selection: Select numeric columns and drop non-predictive IDs/scores
X = df.select_dtypes(include=[np.number]).drop(columns=['kepid', 'koi_score'], errors='ignore')
y = df['koi_disposition'].map({'CONFIRMED': 1, 'FALSE POSITIVE': 0})

# CRITICAL STEP: Drop columns that are completely empty (all NaN values)
# This removes 'koi_teq_err1' and 'koi_teq_err2' cleanly at the beginning
X = X.dropna(how='all', axis=1)

# Split for final visualization and SHAP analysis
X = X.drop(columns=['rowid'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Preprocessing pipeline components
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

In [ ]:
# 2. Define Models for Comparison
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1),
    'CatBoost': CatBoostClassifier(verbose=0, random_state=42)
}

In [ ]:
# 3. Cross-Validation Comparison
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

print("Starting Model Evaluation with 5-Fold Cross Validation...\n")

for name, model in models.items():
    pipeline = Pipeline([
        ('imputer', imputer),
        ('scaler', scaler),
        ('classifier', model)
    ])
    
    scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
    cv_results = cross_validate(pipeline, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    
    results.append({
        'Model': name,
        'Accuracy': cv_results['test_accuracy'].mean(),
        'Precision': cv_results['test_precision'].mean(),
        'Recall': cv_results['test_recall'].mean(),
        'F1-Score': cv_results['test_f1'].mean(),
        'ROC-AUC': cv_results['test_roc_auc'].mean()
    })
    print(f"✓ {name} completed.")

# Create results DataFrame and sort by ROC-AUC
df_results = pd.DataFrame(results).sort_values(by='ROC-AUC', ascending=False)
print("\n### Model Performance Summary ###")
print(df_results.to_markdown(index=False))

In [ ]:
# 4. Multi-Model ROC Curve Visualization
plt.figure(figsize=(10, 7))
for name, model in models.items():
    # Fit pipeline on training set
    pipeline = Pipeline([('imputer', imputer), ('scaler', scaler), ('clf', model)])
    pipeline.fit(X_train, y_train)
    
    # Predict probabilities for the test set
    y_score = pipeline.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_auc = auc(fpr, tpr)
    
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison - Exoplanet Detection')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
# Save ROC Curve to results directory
plt.savefig('../results/roc_curve_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 5. Interpretability with SHAP (Using the best performing model - XGBoost as example)
print("\nCalculating SHAP values for XGBoost...")

# Manually preprocess X_train and X_test for SHAP consistency
X_train_preprocessed = pd.DataFrame(
    scaler.fit_transform(imputer.fit_transform(X_train)),
    columns=X_train.columns
)
X_test_preprocessed = pd.DataFrame(
    scaler.transform(imputer.transform(X_test)), # Use transform, not fit_transform for test set!
    columns=X_test.columns
)

# Fit the standalone model for SHAP
best_model = XGBClassifier(eval_metric='logloss', random_state=42)
best_model.fit(X_train_preprocessed, y_train)

# Calculate SHAP values
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_preprocessed)

In [ ]:
# Plot 1: SHAP Feature Importance (Bar plot)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_preprocessed, plot_type="bar", show=False)
plt.title('SHAP Feature Importance (XGBoost)')
plt.savefig('../results/shap_summary_bar.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 2: SHAP Summary Plot (Bee swarm plot)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_preprocessed, show=False)
plt.savefig('../results/shap_summary_bee.png', dpi=300, bbox_inches='tight')
plt.show()